# Notebook 02: Universal Preprocessing

**Purpose:** Batch process all audio into standardized format for both classical and neural training

**Key Tasks:**
1. Resample all audio to 16 kHz
2. Apply data augmentation (class-aware multipliers)
3. Peak normalization
4. High-pass filtering (80 Hz)
5. Segmentation (1s segments, 50% overlap)
6. Save processed WAV files
7. Track with DVC and log to MLflow

**Outputs:**
- `data/processed/universal/*.wav`
- `manifest.json` (file paths, labels, segment IDs)
- Augmentation log

---

## 1. Import Libraries

In [1]:
import os
import sys
from pathlib import Path
import json
import yaml
from datetime import datetime
import subprocess

# Audio processing
import librosa
import soundfile as sf
from scipy import signal

# Data manipulation
import numpy as np
import pandas as pd

# MLflow
import mlflow

# Progress tracking
from tqdm import tqdm

print('✓ Libraries imported')
print(f'librosa version: {librosa.__version__}')
print(f'soundfile version: {sf.__version__}')

✓ Libraries imported
librosa version: 0.10.2.post1
soundfile version: 0.13.1


## 2. Project Setup

In [2]:
PROJECT_ROOT = Path('/Users/harryirving/Development/projects/ai-ml/BikeAIv5')
os.chdir(PROJECT_ROOT)

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
LOGS_DIR = PROJECT_ROOT / 'logs'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = {
    'grinder': RAW_DATA_DIR / 'grinder',
    'background': RAW_DATA_DIR / 'background',
    'tools': RAW_DATA_DIR / 'tools'
}

print(f'Project Root: {PROJECT_ROOT}')
print(f'Raw Data: {RAW_DATA_DIR}')
print(f'Processed Output: {PROCESSED_DATA_DIR}')

Project Root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Raw Data: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/raw
Processed Output: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/universal


## 3. Load Configuration Parameters

In [3]:
# Load params.yaml 
with open(PROJECT_ROOT / 'params.yaml', 'r') as f: 
    params = yaml.safe_load(f) 
 
# Extract key parameters 
TARGET_SR = params['preprocessing']['target_sr'] 
SEGMENT_DURATION = params['preprocessing']['segment_duration'] 
SEGMENT_OVERLAP = params['preprocessing']['segment_overlap'] 
HIGHPASS_CUTOFF = params['preprocessing']['highpass_cutoff'] 
PEAK_LEVEL = params['preprocessing']['peak_level'] 
 
# Augmentation parameters 
AUG_MULTIPLIERS = { 
    'grinder': params['augmentation']['grinder_multiplier'], 
    'tools': params['augmentation']['tools_multiplier'], 
    'background': params['augmentation']['background_multiplier'] 
} 
 
NOISE_SNR_RANGE = (params['augmentation']['noise_snr_min'], params['augmentation']['noise_snr_max']) 
TIME_STRETCH_RANGE = (params['augmentation']['time_stretch_min'], params['augmentation']['time_stretch_max']) 
PITCH_SHIFT_RANGE = (params['augmentation']['pitch_shift_min'], params['augmentation']['pitch_shift_max']) 
GAIN_RANGE = (params['augmentation']['gain_min'], params['augmentation']['gain_max']) 
 
print('Configuration Loaded:') 
print(f'  Target SR: {TARGET_SR} Hz') 
print(f'  Segment: {SEGMENT_DURATION}s with {SEGMENT_OVERLAP*100}% overlap') 
print(f'  Augmentation multipliers: {AUG_MULTIPLIERS}') 
print('✓ Parameters loaded')

Configuration Loaded:
  Target SR: 16000 Hz
  Segment: 1.0s with 50.0% overlap
  Augmentation multipliers: {'grinder': 11, 'tools': 9, 'background': 1}
✓ Parameters loaded


## 4. Setup MLflow

In [4]:
mlflow.set_tracking_uri(f'file://{LOGS_DIR / "mlruns"}')
mlflow.set_experiment('angle_grinder_pipeline')
print('✓ MLflow configured')

✓ MLflow configured


## 5. Scan Raw Audio Files

In [5]:
def find_audio_files(directory):
    audio_extensions = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
    if not directory.exists():
        return []
    return sorted([f for ext in audio_extensions for f in directory.glob(f'*{ext}')])

audio_files_by_class = {cls: find_audio_files(dir) for cls, dir in CLASS_DIRS.items()}

print('='*50)
total_files = 0
for class_name, files in audio_files_by_class.items():
    count = len(files)
    total_files += count
    print(f'{class_name}: {count} files (x{AUG_MULTIPLIERS[class_name]} aug)')
print('='*50)
print(f'Total raw files: {total_files}')

if total_files == 0:
    raise ValueError('No audio files found! Add files to data/raw/ first.')

grinder: 36 files (x11 aug)
background: 1348 files (x1 aug)
tools: 21 files (x9 aug)
Total raw files: 1405


## 6. Define Preprocessing Functions\n
 
Core audio processing operations.

In [6]:
def resample_audio(audio, orig_sr, target_sr): 
    """Resample audio to target sample rate.""" 
    if orig_sr != target_sr: 
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=target_sr) 
    return audio 
 
def normalize_peak(audio, peak_level=0.89): 
    """Peak normalization to avoid clipping.""" 
    max_amp = np.max(np.abs(audio)) 
    if max_amp > 0: 
        audio = audio / max_amp * peak_level 
    return audio 
 
def highpass_filter(audio, sr, cutoff=80, order=5): 
    """ Apply high-pass filter to remove low-frequency rumble""" 
    nyquist = sr / 2 
    normal_cutoff = cutoff / nyquist 
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False) 
    filtered = signal.filtfilt(b, a, audio) 
    return filtered 
 
def segment_audio(audio, sr, segment_duration=1.0, overlap=0.5): 
    """ Segment audio into fixed-length chunks with overlap""" 
    window_size = int(segment_duration * sr) 
    hop_length = int(window_size * (1 - overlap)) 
     
    segments = [] 
    for start in range(0, len(audio) - window_size + 1, hop_length): 
        segment = audio[start:start + window_size] 
        segments.append(segment) 
     
    return segments 
 
print('✓ Preprocessing functions defined')

✓ Preprocessing functions defined


## 7. Define Augmentation Functions 
 
Data augmentation to increase dataset size.

In [7]:
def add_noise(audio, snr_db_range): 
    """Add white noise at specified SNR. """ 
    snr_db = np.random.uniform(*snr_db_range) 
    audio_power = np.mean(audio ** 2) 
    noise_power = audio_power / (10 ** (snr_db / 10)) 
    noise = np.random.normal(0, np.sqrt(noise_power), len(audio)) 
    return audio + noise 
 
def time_stretch(audio, rate_range): 
    """Time stretch without changing pitch. """ 
    rate = np.random.uniform(*rate_range) 
    return librosa.effects.time_stretch(audio, rate=rate) 
 
def pitch_shift(audio, sr, n_steps_range): 
    """Shift pitch by random semitones. """ 
    n_steps = np.random.uniform(*n_steps_range) 
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps) 
 
def apply_gain(audio, gain_db_range): 
    """Apply random gain adjustment. """ 
    gain_db = np.random.uniform(*gain_db_range) 
    gain_linear = 10 ** (gain_db / 20) 
    return audio * gain_linear 
 
def augment_audio(audio, sr, class_name, aug_params): 
    """Apply random augmentations to audio. """ 
    # Only augment grinder and tools (not background) 
    if class_name == 'background': 
        return audio, {} 
     
    aug_log = {} 
     
    # Apply augmentations 
    audio = add_noise(audio, aug_params['noise_snr']) 
    aug_log['noise_applied'] = True 
     
    audio = time_stretch(audio, aug_params['time_stretch']) 
    aug_log['time_stretch_applied'] = True 
     
    audio = pitch_shift(audio, sr, aug_params['pitch_shift']) 
    aug_log['pitch_shift_applied'] = True 
     
    audio = apply_gain(audio, aug_params['gain']) 
    aug_log['gain_applied'] = True 
     
    return audio, aug_log 
 
print('✓ Augmentation functions defined')

✓ Augmentation functions defined


## 8. Main Processing Pipeline 
 
Process all audio files with augmentation and segmentation.

In [8]:
def process_file(filepath, class_name, file_idx, aug_idx, aug_params): 
    """Process a single audio file through the pipeline. """ 
    # Load audio 
    audio, orig_sr = librosa.load(filepath, sr=None) 
     
    # Resample 
    audio = resample_audio(audio, orig_sr, TARGET_SR) 
     
    # Apply augmentation (if not original) 
    if aug_idx > 0: 
        audio, aug_log = augment_audio(audio, TARGET_SR, class_name, aug_params) 
    else: 
        aug_log = {'original': True} 
     
    # Normalize 
    audio = normalize_peak(audio, PEAK_LEVEL) 
     
    # High-pass filter 
    audio = highpass_filter(audio, TARGET_SR, HIGHPASS_CUTOFF) 
     
    # Segment 
    segments = segment_audio(audio, TARGET_SR, SEGMENT_DURATION, SEGMENT_OVERLAP) 
     
    # Save segments 
    segment_info = [] 
    for seg_idx, segment in enumerate(segments): 
        # Create filename: class_fileXX_augYY_segZZ.wav 
        filename = f'{class_name}_{file_idx:04d}_{aug_idx:02d}_{seg_idx:04d}.wav' 
        output_path = PROCESSED_DATA_DIR / filename 
         
        # Save WAV 
        sf.write(output_path, segment, TARGET_SR) 
         
        # Record metadata 
        segment_info.append({ 
            'filepath': str(output_path.relative_to(PROJECT_ROOT)), 
            'class': class_name, 
            'label': 1 if class_name == 'grinder' else 0, 
            'original_file': filepath.name, 
            'file_idx': file_idx, 
            'aug_idx': aug_idx, 
            'segment_idx': seg_idx, 
            'augmentation': aug_log 
        }) 
     
    return segment_info 
 
print('✓ Processing pipeline defined')

✓ Processing pipeline defined


## 9. Process All Audio Files 
 
Main processing loop with augmentation multipliers.

In [9]:
# Prepare augmentation parameters 
aug_params = { 
    'noise_snr': NOISE_SNR_RANGE, 
    'time_stretch': TIME_STRETCH_RANGE, 
    'pitch_shift': PITCH_SHIFT_RANGE, 
    'gain': GAIN_RANGE 
} 
 
manifest = [] 
augmentation_log = [] 
 
print('Starting audio processing...') 
print('='*70) 
 
for class_name, files in audio_files_by_class.items(): 
    if len(files) == 0: 
        continue 
     
    multiplier = AUG_MULTIPLIERS[class_name] 
    print(f' Processing {class_name} ({len(files)} files x {multiplier} augmentations)...') 
     
    for file_idx, filepath in enumerate(tqdm(files, desc=f'  {class_name}')): 
        try: 
            # Process original + augmented versions 
            for aug_idx in range(multiplier): 
                segment_info = process_file(filepath, class_name, file_idx, aug_idx, aug_params) 
                manifest.extend(segment_info) 
                 
                augmentation_log.append({ 
                    'original_file': filepath.name, 
                    'class': class_name, 
                    'aug_idx': aug_idx, 
                    'num_segments': len(segment_info) 
                }) 
        except Exception as e: 
            print(f'   ⚠️  Failed to process {filepath.name}: {e}') 
 
print('\n' + '='*70) 
print(f'✓ Processing complete!') 
print(f'  Total segments created: {len(manifest)}')

Starting audio processing...
 Processing grinder (36 files x 11 augmentations)...


  grinder: 100%|██████████| 36/36 [01:35<00:00,  2.65s/it]


 Processing background (1348 files x 1 augmentations)...


  background: 100%|██████████| 1348/1348 [00:03<00:00, 342.28it/s]


 Processing tools (21 files x 9 augmentations)...


  tools: 100%|██████████| 21/21 [00:54<00:00,  2.60s/it]


✓ Processing complete!
  Total segments created: 60324


## 10. Processing Statistics

In [10]:
# Convert manifest to DataFrame 
df_manifest = pd.DataFrame(manifest) 
 
print(' PROCESSING STATISTICS') 
print('='*70) 
print(f'Total segments: {len(df_manifest)}') 
print(' Segments by class:') 
for class_name, count in df_manifest['class'].value_counts().items(): 
    percentage = (count / len(df_manifest)) * 100 
    print(f'  {class_name:15s}: {count:>6} segments ({percentage:>5.1f}%)') 
print(' Label distribution:') 
for label, count in df_manifest['label'].value_counts().items(): 
    label_name = 'grinder' if label == 1 else 'non-grinder' 
    print(f'  {label_name:15s}: {count:>6} segments') 
print('='*70)

 PROCESSING STATISTICS
Total segments: 60324
 Segments by class:
  grinder        :  32972 segments ( 54.7%)
  tools          :  19283 segments ( 32.0%)
  background     :   8069 segments ( 13.4%)
 Label distribution:
  grinder        :  32972 segments
  non-grinder    :  27352 segments


## 11. Save Manifest and Logs

In [11]:
# Save manifest 
manifest_path = PROJECT_ROOT / 'data' / 'processed' / 'manifest.json' 
with open(manifest_path, 'w') as f: 
    json.dump(manifest, f, indent=2) 
print(f'✓ Saved manifest: {manifest_path}') 
 
# Save augmentation log 
aug_log_path = PROJECT_ROOT / 'data' / 'processed' / 'augmentation_log.json' 
with open(aug_log_path, 'w') as f: 
    json.dump(augmentation_log, f, indent=2) 
print(f'✓ Saved augmentation log: {aug_log_path}') 
 
# Save manifest as CSV for easier viewing 
csv_path = PROJECT_ROOT / 'data' / 'processed' / 'manifest.csv' 
df_manifest.to_csv(csv_path, index=False) 
print(f'✓ Saved manifest CSV: {csv_path}')

✓ Saved manifest: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/manifest.json
✓ Saved augmentation log: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/augmentation_log.json
✓ Saved manifest CSV: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/manifest.csv


## 12. Track Processed Data with DVC 
 
Add processed data to DVC version control.

In [12]:
print(' Tracking with DVC...') 
 
# Add processed data to DVC 
result = subprocess.run( 
    ['dvc', 'add', 'data/processed'], 
    cwd=PROJECT_ROOT, 
    capture_output=True, 
    text=True 
) 
 
if result.returncode == 0: 
    print('✓ Added data/processed to DVC') 
    print(' Next steps:') 
    print('  git add data/processed.dvc .gitignore') 
    print('  git commit -m "Add processed audio data"') 
    print('  dvc push') 
else: 
    print(f'Note: {result.stderr}')

 Tracking with DVC...


KeyboardInterrupt: 

## 13. Log to MLflow

In [ ]:
with mlflow.start_run(run_name='02_universal_preprocessing'): 
    # Log parameters 
    mlflow.log_param('target_sr', TARGET_SR) 
    mlflow.log_param('segment_duration', SEGMENT_DURATION) 
    mlflow.log_param('segment_overlap', SEGMENT_OVERLAP) 
    mlflow.log_param('highpass_cutoff', HIGHPASS_CUTOFF) 
    mlflow.log_param('grinder_multiplier', AUG_MULTIPLIERS['grinder']) 
    mlflow.log_param('tools_multiplier', AUG_MULTIPLIERS['tools']) 
    mlflow.log_param('background_multiplier', AUG_MULTIPLIERS['background']) 
     
    # Log metrics 
    mlflow.log_metric('total_segments', len(df_manifest)) 
    for class_name, count in df_manifest['class'].value_counts().items():
        mlflow.log_metric(f'segments_{class_name}', count)
    
    # Log tags 
    mlflow.set_tags({
        'stage': 'preprocessing',
        'notebook': '02',
        'timestamp': datetime.now().isoformat()
    })
    
    # Log artifacts
    mlflow.log_artifact(str(manifest_path))
    mlflow.log_artifact(str(aug_log_path))
    mlflow.log_artifact(str(csv_path))
    
    print(' ✓ Logged to MLflow')

 ✓ Logged to MLflow


## 14. Summary and Next Steps

In [13]:
print('\n' + '='*70)
print('UNIVERSAL PREPROCESSING COMPLETE')
print('='*70)
print(f' ✓ Processed {len(df_manifest)} segments')
print(f'✓ Saved to: {PROCESSED_DATA_DIR}')
print(f'✓ Manifest: {manifest_path}')
print(f'Class distribution:')
for cls, cnt in df_manifest['class'].value_counts().items():
    print(f'  {cls}: {cnt} segments')
print(f'All audio now:')
print(f'  - Resampled to {TARGET_SR} Hz')
print(f'  - Augmented with class-aware multipliers')
print(f'  - Normalized (peak {PEAK_LEVEL})')
print(f'  - High-pass filtered ({HIGHPASS_CUTOFF} Hz)')
print(f'  - Segmented ({SEGMENT_DURATION}s, {SEGMENT_OVERLAP*100}% overlap)')
print('Next Steps:')
print('1. Track with DVC:')
print('   git add data/processed.dvc')
print('   git commit -m "Add processed data"')
print('   dvc push')
print('2. Proceed to feature extraction:')
print('   → 03_classical_feature_extraction.ipynb (for SVM, RF, XGBoost)')
print('   → 04_neural_feature_extraction.ipynb (for CNNs)')
print('\n' + '='*70)
print('✓ Notebook 02 Complete!')


UNIVERSAL PREPROCESSING COMPLETE
 ✓ Processed 60324 segments
✓ Saved to: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/universal
✓ Manifest: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/processed/manifest.json
Class distribution:
  grinder: 32972 segments
  tools: 19283 segments
  background: 8069 segments
All audio now:
  - Resampled to 16000 Hz
  - Augmented with class-aware multipliers
  - Normalized (peak 0.89)
  - High-pass filtered (80 Hz)
  - Segmented (1.0s, 50.0% overlap)
Next Steps:
1. Track with DVC:
   git add data/processed.dvc
   git commit -m "Add processed data"
   dvc push
2. Proceed to feature extraction:
   → 03_classical_feature_extraction.ipynb (for SVM, RF, XGBoost)
   → 04_neural_feature_extraction.ipynb (for CNNs)

✓ Notebook 02 Complete!
